In [3]:
import os
from typing import Annotated
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver


In [4]:

# 1. Load API keys from environment
load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_NAME="gpt-4.1_1"

# 2. Define the LLM
# Using gpt-4o-mini as a fast, cost-effective default chat model
model = AzureChatOpenAI(azure_endpoint=AZURE_OPENAI_ENDPOINT,
api_version=AZURE_OPENAI_API_VERSION, azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME, temperature=0.2,)


In [8]:

# 3. Define the chatbot node
# MessagesState automatically manages an array of messages using 'add_messages'
def chatbot_node(state: MessagesState):
    """
    Receives the current list of messages, feeds them to the LLM,
    and returns the updated message list.
    """
    response = model.invoke(state["messages"])
    return {"messages": [response]}


# 4. Construct the LangGraph workflow
graph = StateGraph(MessagesState)

# Add our node to the graph layout
graph.add_node("chatbot", chatbot_node)

# Create execution paths
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)


In [9]:
# 5. Compile the graph with memory checkpointer
# MemorySaver acts as an in-memory database keeping track of chat session histories
memory = MemorySaver()
workflow = graph.compile(checkpointer=memory)

In [10]:

# 6. Run an interactive terminal chat loop
if __name__ == "__main__":
    print("🤖 LangGraph Chatbot Initialized! Type 'exit' or 'quit' to end.")
    print("-" * 60)

    # Each distinct conversation thread needs a unique thread_id for its memory
    config = {"configurable": {"thread_id": "user_session_abc123"}}

    while True:
        try:
            user_input = input("\nYou: ").strip()
            if user_input.lower() in ["exit", "quit", "q"]:
                print("Goodbye!")
                break

            if not user_input:
                continue

            # Format input as a user message and pass it down the graph
            events = workflow.stream(
                {"messages": [("user", user_input)]},
                config,
                stream_mode="values"
            )

            # Print only the latest response appended to the message state
            for event in events:
                latest_message = event["messages"][-1]

            # If the latest message came from the AI node, print it
            if latest_message.type == "ai":
                print(f"Bot: {latest_message.content}")

        except KeyboardInterrupt:
            print("\nGoodbye!")
            break
        except Exception as e:
            print(f"\nAn error occurred: {e}")


🤖 LangGraph Chatbot Initialized! Type 'exit' or 'quit' to end.
------------------------------------------------------------
Bot: Sachin Tendulkar is a legendary former Indian cricketer, widely regarded as one of the greatest batsmen in the history of cricket. Born on April 24, 1973, in Mumbai, India, he made his debut for the Indian national team in 1989 at the age of 16.

**Key facts about Sachin Tendulkar:**
- **Nicknames:** "Master Blaster," "Little Master," and "God of Cricket."
- **Career Span:** 1989–2013.
- **Records:** He holds numerous cricket records, including being the highest run-scorer in both Test and One Day International (ODI) cricket.
- **Centuries:** First player to score 100 international centuries (51 in Tests and 49 in ODIs).
- **Milestones:** First cricketer to score a double century (200 runs) in an ODI match.
- **World Cup:** Played six World Cups and was part of the Indian team that won the ICC Cricket World Cup in 2011.
- **Awards:** Received many honors, inc